# APX1145 Frequency-Drop Hypothesis Check

This notebook is intentionally focused on one question:

**Does the APX1145 power drop coincide with a frequency drop?**

It uses only detailed run artifacts (`turbostat` + `perf`) from the latest detailed collection run.

The last cell also draws the paper's thermal figure (APX1186 package power and temperature, all loops overlaid).


In [ ]:
import os
MACHINE = os.environ.get("GREENSQL_MACHINE", "thinkpad")  # "thinkpad" or "dell"
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)

DATA_DIR = Path(f"../runner/logs/detailed_perf_runs_{MACHINE}")

# Analyze these run IDs (add/remove IDs as needed).
RUN_IDS = sorted({p.name.split("__")[0] for p in DATA_DIR.glob("*__detailed_perf_metadata.txt")})

run_meta_files = {}
for rid in RUN_IDS:
    meta_files = sorted(DATA_DIR.glob(f"{rid}__detailed_perf_metadata.txt"), key=lambda p: p.stat().st_mtime)
    if not meta_files:
        raise FileNotFoundError(f"No detailed metadata files found for run_id={rid} under {DATA_DIR}")
    run_meta_files[rid] = meta_files[-1]

print("Run IDs to analyze:", RUN_IDS)
for rid in RUN_IDS:
    print(f"Using metadata for {rid}: {run_meta_files[rid].name}")

In [ ]:
def parse_meta(path: Path) -> dict:
    meta = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            meta[k.strip()] = v.strip()
    return meta


def parse_perf_file(path: Path) -> pd.DataFrame:
    cols = [
        "time_sec", "value", "unit", "event", "runtime_ns", "pct_running", "metric_value", "metric_unit"
    ]
    df = pd.read_csv(path, comment="#", header=None, names=cols)
    df = df.dropna(subset=["event", "value"]).copy()

    m = re.match(r"(?P<run_id>[A-F0-9]+)__(?P<query>.+)\.perf\.csv$", path.name)
    df["run_id"] = m.group("run_id") if m else "unknown"
    df["query"] = m.group("query") if m else path.stem
    df["file_name"] = path.name

    for col in ["time_sec", "value", "runtime_ns", "pct_running", "metric_value"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["sample_idx"] = df.groupby(["query", "file_name", "event"]).cumcount()
    return df


def parse_turbo_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", engine="python")
    m = re.match(r"(?P<run_id>[A-F0-9]+)__(?P<query>.+)\.turbostat\.csv$", path.name)

    if "Core" in df.columns:
        df = df[df["Core"].astype(str) != "Core"].copy()

    df["run_id"] = m.group("run_id") if m else "unknown"
    df["query"] = m.group("query") if m else path.stem
    df["file_name"] = path.name

    num_cols = ["PkgWatt", "CorWatt", "RAMWatt", "PkgTmp", "CoreTmp", "Avg_MHz", "Bzy_MHz", "Busy%", "IPC", "TSC_MHz"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Core_str"] = df.get("Core", pd.Series([np.nan] * len(df))).astype(str)
    df["CPU_str"] = df.get("CPU", pd.Series([np.nan] * len(df))).astype(str)

    package_mask = (df["Core_str"] == "-") & (df["CPU_str"] == "-")
    if package_mask.any():
        df["sample_id"] = package_mask.cumsum() - 1
    else:
        df["sample_id"] = np.arange(len(df))

    df["sample_id"] = pd.to_numeric(df["sample_id"], errors="coerce")
    return df


run_frames = []
for rid in RUN_IDS:
    meta = parse_meta(run_meta_files[rid])
    run_id = meta.get("run_id", rid)
    sample_interval_sec = int(meta.get("sample_interval_sec", "1") or 1)

    perf_files = sorted(DATA_DIR.glob(f"{run_id}__*.perf.csv"))
    turbo_files = sorted(DATA_DIR.glob(f"{run_id}__*.turbostat.csv"))

    if not perf_files or not turbo_files:
        raise FileNotFoundError(f"Run {run_id} does not have both perf and turbostat files")

    perf_df = pd.concat([parse_perf_file(p) for p in perf_files], ignore_index=True)
    turbo_df = pd.concat([parse_turbo_file(p) for p in turbo_files], ignore_index=True)
    turbo_df["time_s"] = turbo_df["sample_id"] * sample_interval_sec

    run_frames.append({
        "run_id": run_id,
        "sample_interval_sec": sample_interval_sec,
        "perf_df": perf_df,
        "turbo_df": turbo_df,
    })

print(f"Loaded {len(run_frames)} run(s)")
for frame in run_frames:
    print(
        f"run_id={frame['run_id']} | perf_rows={len(frame['perf_df']):,} | turbostat_rows={len(frame['turbo_df']):,} | interval={frame['sample_interval_sec']}s"
    )

In [ ]:
def detect_drop_pos(series: pd.Series, min_frac: float = 0.2, window: int = 9) -> int:
    s = series.reset_index(drop=True).astype(float).dropna()
    n = len(s)
    if n < 20:
        return int(max(1, n // 2))

    smooth = s.rolling(window=window, min_periods=1, center=True).median().reset_index(drop=True)

    start = max(window, int(n * min_frac))
    end = min(n - window - 1, int(n * (1 - min_frac)))
    if end <= start:
        return int(n // 2)

    best_pos = start
    best_score = -np.inf
    for i in range(start, end + 1):
        pre_med = smooth.iloc[max(0, i - window):i].median()
        post_med = smooth.iloc[i:min(n, i + window)].median()
        score = pre_med - post_med
        if pd.notna(score) and score > best_score:
            best_score = score
            best_pos = i

    return int(best_pos)


run_results = []
for frame in run_frames:
    run_id = frame["run_id"]
    sample_interval_sec = frame["sample_interval_sec"]
    turbo_df = frame["turbo_df"]

    pkg = turbo_df[(turbo_df["Core_str"] == "-") & (turbo_df["CPU_str"] == "-")].copy()
    if pkg.empty:
        use_cols = [c for c in ["PkgWatt", "Avg_MHz", "Bzy_MHz", "Busy%", "IPC"] if c in turbo_df.columns]
        pkg = turbo_df.groupby(["query", "sample_id"], as_index=False)[use_cols].median()
    else:
        pkg = pkg.sort_values(["query", "sample_id"]).copy()

    pkg["time_s"] = pkg["sample_id"] * sample_interval_sec

    drop_points = {}
    for query, g in pkg.groupby("query"):
        g = g.sort_values("sample_id").reset_index(drop=True)
        if "PkgWatt" in g.columns and g["PkgWatt"].notna().sum() > 0:
            pos = detect_drop_pos(g["PkgWatt"])
            drop_points[query] = int(g.loc[pos, "sample_id"])
        else:
            drop_points[query] = int(g["sample_id"].median())

    pkg["phase"] = "pre-drop"
    for query, sample_cut in drop_points.items():
        mask = (pkg["query"] == query) & (pkg["sample_id"] >= sample_cut)
        pkg.loc[mask, "phase"] = "post-drop"

    run_results.append({
        "run_id": run_id,
        "sample_interval_sec": sample_interval_sec,
        "turbo_df": turbo_df,
        "pkg": pkg,
        "drop_points": drop_points,
    })

for result in run_results:
    print(f"run_id={result['run_id']} drop points (sample_id): {result['drop_points']}")
    display(result["pkg"].groupby(["query", "phase"], as_index=False).size())

In [ ]:
# Plot 1: package-level signals per base query (A vs B), one figure per metric.
# Faceted small multiples; thin line = raw samples, thick line = rolling median (9 s).
import math
plots_dir = Path("plots"); plots_dir.mkdir(parents=True, exist_ok=True)
VARIANT_COLORS = {"A": "#1f77b4", "B": "#ff7f0e"}

def add_base_variant(df):
    df = df.copy()
    ex = df["query"].str.extract(r"^(?P<base>.+)-query(?P<variant>[AB]).sql$")
    df["base"], df["variant"] = ex["base"], ex["variant"]
    return df.dropna(subset=["base", "variant"])

def facet_ab(df, ycol, title, ylabel, drop_points=None, interval=1, ncols=4, save_as=None):
    df = add_base_variant(df)
    bases = sorted(df["base"].unique())
    if not bases:
        print(f"{title}: no A/B queries"); return
    nrows = math.ceil(len(bases) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 2.9 * nrows), squeeze=False)
    for ax in axes.ravel()[len(bases):]:
        ax.set_visible(False)
    for ax, base in zip(axes.ravel(), bases):
        for variant in ["A", "B"]:
            g = df[(df["base"] == base) & (df["variant"] == variant)].sort_values("time_s")
            if g.empty:
                continue
            col = VARIANT_COLORS[variant]
            ax.plot(g["time_s"], g[ycol], color=col, alpha=0.25, linewidth=0.7)
            ax.plot(g["time_s"], g[ycol].rolling(9, min_periods=1, center=True).median(), color=col, linewidth=1.5, label=f"query{variant}")
            q = g["query"].iloc[0]
            if drop_points and q in drop_points:
                ax.axvline(drop_points[q] * interval, color=col, linestyle="--", linewidth=0.9, alpha=0.8)
        ax.set_title(base, fontsize=10)
        ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("Time in run (s)")
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel)
    h, l = next((a.get_legend_handles_labels() for a in axes.ravel() if a.get_legend_handles_labels()[0]), ([], []))
    fig.legend(h, l, loc="upper right", ncol=2)
    fig.suptitle(title, fontsize=12, y=1.0)
    plt.tight_layout()
    if save_as:
        fig.savefig(plots_dir / f"{save_as}.svg", format="svg", bbox_inches="tight")
    plt.show()

for result in run_results:
    run_id = result["run_id"]
    pkg = result["pkg"]
    for metric in [c for c in ["PkgWatt", "Avg_MHz", "Bzy_MHz", "Busy%", "IPC", "PkgTmp"] if c in pkg.columns and pkg[c].notna().any()]:
        facet_ab(pkg, metric, f"{MACHINE} | run_id={run_id} | {metric} over time (dashed = detected drop)", metric,
                 drop_points=result["drop_points"], interval=result["sample_interval_sec"],
                 save_as=f"detailed_{MACHINE}_{metric.replace('%', 'pct')}")


In [ ]:
# Plot 2: per-CPU Avg_MHz over time for the APX1145 queryA/B pair (the frequency-drop hypothesis).
for result in run_results:
    run_id = result["run_id"]
    cpu_df = result["turbo_df"].copy()
    cpu_df["cpu_id"] = pd.to_numeric(cpu_df["CPU_str"], errors="coerce")
    cpu_df = cpu_df.dropna(subset=["cpu_id"]).copy()
    cpu_df["cpu_id"] = cpu_df["cpu_id"].astype(int)
    cpu_df = cpu_df[cpu_df["query"].str.match(r"^APX1145-query[AB]")]
    if cpu_df.empty or cpu_df["Avg_MHz"].notna().sum() == 0:
        print(f"run_id={run_id}: no per-CPU Avg_MHz for APX1145"); continue
    cpu_df["time_s"] = cpu_df["sample_id"] * result["sample_interval_sec"]
    cpus = sorted(cpu_df["cpu_id"].unique())
    ncols = 2
    nrows = math.ceil(len(cpus) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 2.6 * nrows), squeeze=False, sharex=True, sharey=True)
    for ax in axes.ravel()[len(cpus):]:
        ax.set_visible(False)
    for ax, cpu in zip(axes.ravel(), cpus):
        for query, g in cpu_df[cpu_df["cpu_id"] == cpu].groupby("query"):
            g = g.sort_values("sample_id")
            variant = "A" if "queryA" in query else "B"
            ax.plot(g["time_s"], g["Avg_MHz"], color=VARIANT_COLORS[variant], alpha=0.25, linewidth=0.7)
            ax.plot(g["time_s"], g["Avg_MHz"].rolling(9, min_periods=1, center=True).median(), color=VARIANT_COLORS[variant], linewidth=1.5, label=query)
            if query in result["drop_points"]:
                ax.axvline(result["drop_points"][query] * result["sample_interval_sec"], color=VARIANT_COLORS[variant], linestyle="--", linewidth=0.9)
        ax.set_title(f"CPU {cpu}", fontsize=10); ax.grid(alpha=0.3)
    for ax in axes[:, 0]: ax.set_ylabel("Avg_MHz")
    for ax in axes[-1]: ax.set_xlabel("Time in run (s)")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper right", ncol=2)
    fig.suptitle(f"{MACHINE} | run_id={run_id} | APX1145: per-CPU Avg_MHz", y=1.0)
    plt.tight_layout()
    fig.savefig(plots_dir / f"detailed_{MACHINE}_APX1145_per_cpu_mhz.svg", format="svg", bbox_inches="tight")
    plt.show()


In [ ]:
# Total Avg_MHz summed over all CPUs, per base query (A vs B).
for result in run_results:
    cpu_df = result["turbo_df"].copy()
    cpu_df["cpu_id"] = pd.to_numeric(cpu_df["CPU_str"], errors="coerce")
    cpu_df = cpu_df.dropna(subset=["cpu_id"])
    if "Avg_MHz" not in cpu_df.columns or cpu_df["Avg_MHz"].notna().sum() == 0:
        print("Avg_MHz not available"); continue
    total = cpu_df.groupby(["query", "sample_id"], as_index=False)["Avg_MHz"].sum().rename(columns={"Avg_MHz": "Total_Avg_MHz"})
    total["time_s"] = total["sample_id"] * result["sample_interval_sec"]
    facet_ab(total, "Total_Avg_MHz", f"{MACHINE} | run_id={result['run_id']} | total MHz (sum of per-CPU Avg_MHz)", "Total Avg_MHz",
             drop_points=result["drop_points"], interval=result["sample_interval_sec"], save_as=f"detailed_{MACHINE}_total_mhz")


In [ ]:
# Paper figure (thermal throttling): package power and temperature, all loops overlaid by time-in-loop.
# Thin lines = individual loops, thick = median. One single-column figure per machine and guideline.
import importlib
import figure_style as fs   # shared palette + paper-figure functions (analysis/figure_style.py)
importlib.reload(fs)

THERMAL_FAMILIES = ["APX1186"]      # add e.g. "APX1145" to draw more guidelines
detailed_runs = pd.read_csv(f"../runner/logs/detailed_perf_runs_{MACHINE}.csv")
machine_label = fs.PC_NAME["PC_1" if MACHINE == "thinkpad" else "PC_2"]

for result in run_results:
    run_id = result["run_id"]
    pkg = result["pkg"]
    if "PkgWatt" not in pkg.columns or "PkgTmp" not in pkg.columns:
        print(f"run_id={run_id}: PkgWatt/PkgTmp not available"); continue
    for fam in THERMAL_FAMILIES:
        parts, n_loops = {}, None
        for v in "AB":
            test = f"{fam}-query{v}.sql"
            g = pkg[pkg["query"] == test].sort_values("sample_id")
            if g.empty:
                break
            parts[v] = g[["PkgWatt", "PkgTmp"]].reset_index(drop=True)
            rows = detailed_runs[(detailed_runs["run_id"].astype(str) == str(run_id)) & (detailed_runs["test_name"] == test)]
            n_loops = int(rows["loop_index"].nunique())
        if len(parts) == 2 and n_loops:
            fs.thermal_overlaid(parts, n_loops, fam, machine_label, f"thermal_{fam.lower()}_{MACHINE}")
        else:
            print(f"run_id={run_id}: no {fam} A/B rows")
